In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBRegressor

df = pd.read_csv('./Preprocessing/Feature_Encoded_data.csv')
df.drop('Unnamed: 0.1', axis=1, inplace=True)
df.drop('Unnamed: 0', axis=1, inplace=True)
df.drop('Id', axis=1, inplace=True)
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_grid = {
    'max_depth': [3, 4, 5],
    'min_child_weight': [3, 5, 7],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
}

#train with grid search to find best hyperparameters
grid_search = GridSearchCV(
    estimator=XGBRegressor(
        n_estimators = 300,
        learning_rate = 0.05,
        reg_alpha = 0.1,
        reg_lambda = 1.0,
        random_state = 42
    ),
    param_grid = param_grid,
    cv = 5,
    scoring = 'r2',
    verbose = 1,
    n_jobs = -1
)

grid_search.fit(X_train, y_train)
print(f"best_params: {grid_search.best_params_}")

#train with best hyperparameters found and early stopping
best_params = grid_search.best_params_
model = XGBRegressor(
    **best_params,
    n_estimators = 1000,
    learning_rate = 0.05,
    reg_alpha = 0.1,
    reg_lambda = 1.0,
    early_stopping_rounds = 50,
    random_state = 42
)

model.fit(
    X_train, y_train,
    eval_set = [(X_test, y_test)],
    verbose = 100
)

Fitting 5 folds for each of 81 candidates, totalling 405 fits
best_params: {'colsample_bytree': 0.7, 'max_depth': 3, 'min_child_weight': 3, 'subsample': 0.8}
[0]	validation_0-rmse:71561.90595
[100]	validation_0-rmse:22392.79325
[200]	validation_0-rmse:21065.00681
[300]	validation_0-rmse:20727.65088
[365]	validation_0-rmse:20721.63048


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.7
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [2]:
score = model.score(X_test, y_test)
print(f"R2 score: {score}")

R2 score: 0.922454297542572


In [3]:
train_score = model.score(X_train, y_train)
print(f"R2 train score: {train_score}")

R2 train score: 0.9793952107429504


In [4]:
# Load test data
test_df = pd.read_csv('./Data/test.csv')

In [5]:
#checking if any column from test set contains null values
test_df.columns[test_df.isnull().any()]

Index(['MSZoning', 'LotFrontage', 'Alley', 'Utilities', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath',
       'BsmtHalfBath', 'KitchenQual', 'Functional', 'FireplaceQu',
       'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea',
       'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType'],
      dtype='object')

In [6]:
#gotta clean the test data the same way we cleaned the train data
def clean(df):
    #All the changes I made myself
    
    # Make a copy to avoid modifying original
    data = df.copy()
    
    # Drop unnecessary columns (based on EDA: redundant, low variance, or weak correlation)
    columns_to_drop = [
        'Utilities', 'Street', 'Condition2', 'RoofMatl', 
        'BsmtFinSF1', 'BsmtFinSF2', 'Heating', 'LowQualFinSF', 
        '3SsnPorch', 'ScreenPorch', 'PoolArea', 'PoolQC', 
        'MiscFeature', 'MiscVal'
    ]
    data.drop(columns=columns_to_drop, axis=1, inplace=True, errors='ignore')
    
    # Transform KitchenAbvGr: group !=1 into 0 (1 kitchen vs multiple/none)
    data.loc[data['KitchenAbvGr'] != 1, 'KitchenAbvGr'] = 0
    
    # Transform GarageQual: group != 'TA' into 'Other'
    data.loc[data['GarageQual'] != 'TA', 'GarageQual'] = 'Other'
    
    # Transform GarageCond: group != 'TA' into 'Other'
    data.loc[data['GarageCond'] != 'TA', 'GarageCond'] = 'Other'
    
    # Transform EnclosedPorch: group >0 into 1 (has porch vs not)
    data.loc[data['EnclosedPorch'] > 0, 'EnclosedPorch'] = 1
    
    # Transform SaleType: group rare types into 'Other'
    data.loc[~data['SaleType'].isin(['WD', 'New', 'COD']), 'SaleType'] = 'Other'
    
    # Transform SaleCondition: group rare conditions into 'Other'
    data.loc[~data['SaleCondition'].isin(['Normal', 'Partial']), 'SaleCondition'] = 'Other'
    
    return data

def fillna(data):
    # Fill LotFrontage by neighborhood median
    data['LotFrontage'] = data.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

    # Fill Alley with 'None'
    data["Alley"] = data["Alley"].fillna("None")

    # Fill MasVnrType and MasVnrArea
    data['MasVnrType'] = data['MasVnrType'].fillna('None')
    data['MasVnrArea'] = data['MasVnrArea'].fillna(0)

    # Fill basement columns
    data['BsmtQual'] = data['BsmtQual'].fillna('None')
    data['BsmtCond'] = data['BsmtCond'].fillna('None')
    data['BsmtExposure'] = data['BsmtExposure'].fillna('None')
    data['BsmtFinType1'] = data['BsmtFinType1'].fillna('None')
    data['BsmtFinType2'] = data['BsmtFinType2'].fillna('None')

    # Fill Electrical by neighborhood mode
    data['Electrical'] = data.groupby('Neighborhood')['Electrical'].transform(lambda x: x.fillna(x.mode()[0]))

    # Fill garage columns
    data['GarageType'] = data['GarageType'].fillna('None')
    data['GarageYrBlt'] = data['GarageYrBlt'].fillna(0)
    data['GarageFinish'] = data['GarageFinish'].fillna('None')
    data['GarageQual'] = data['GarageQual'].fillna('None')
    data['GarageCond'] = data['GarageCond'].fillna('None')

    # Fill other columns
    data['FireplaceQu'] = data['FireplaceQu'].fillna('None')
    data['Fence'] = data['Fence'].fillna('None')
    
    # Fill additional columns with nulls
    data['BsmtUnfSF'] = data['BsmtUnfSF'].fillna(0)
    data['TotalBsmtSF'] = data['TotalBsmtSF'].fillna(0)
    data['KitchenQual'] = data['KitchenQual'].fillna('TA')
    data['GarageCars'] = data['GarageCars'].fillna(0)
    data['GarageArea'] = data['GarageArea'].fillna(0)
    data['BsmtFullBath'] = data['BsmtFullBath'].fillna(0)
    data['BsmtHalfBath'] = data['BsmtHalfBath'].fillna(0)

    return data

def get_season(month):
    if month in [12,1,2]: 
        return 'Winter'
    elif month in [3,4,5]:
        return 'Spring'
    elif month in [6,7,8]:
        return 'Summer'
    else:
        return 'Fall'
    
def feature_engineering(df):
    #only keep one exterior column
    df['MixedExterior'] = (df['Exterior1st'] == df['Exterior2nd']).astype(int)
    df.drop('Exterior2nd', axis=1, inplace=True)
    #only keep total bath count
    df['TotalBath'] = (df['FullBath'] + df['BsmtFullBath']) + 0.5*(df['HalfBath'] + df['BsmtHalfBath'])
    df.drop('FullBath', axis=1, inplace=True)
    df.drop('BsmtFullBath', axis=1, inplace=True)
    df.drop('HalfBath', axis=1, inplace=True)
    df.drop('BsmtHalfBath', axis=1, inplace=True)
    #house age
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    #year since remodeled
    df['YrSinceRemod'] = df['YrSold'] - df['YearRemodAdd']
    #whether the house was remodeled
    df['Remodeled'] = (df['YearBuilt'] == df['YearRemodAdd'])
    #delete year built and remodeled
    df.drop(['YearBuilt', 'YearRemodAdd'], axis=1, inplace=True)
    #garage condition doesn't matter much, only keep the quality
    df.drop('GarageCond', axis=1, inplace=True)
    #season sold
    df['SeasonSold'] = df['MoSold'].apply(get_season)
    df.drop('MoSold', axis=1, inplace=True)
    #total square feet
    df['TotalSF'] = df['1stFlrSF'] + df['2ndFlrSF'] + df['TotalBsmtSF']
    #new house
    df['NewHouse'] = (df['HouseAge'] < 2).astype(int)
    return df

def feature_encoding(df):
    categorical_cols = df.select_dtypes(include='object').columns.tolist()
    qual_map = {'None': 0, 'Other': 1, 'Po': 1, 'Fa':2, 'TA': 3, 'Gd': 4, 'Ex': 5}
    ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond','HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual']
    for col in ordinal_cols:
        df[col] = df[col].map(qual_map)
    nominal_cols = [col for col in categorical_cols if col not in ordinal_cols]
    df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)
    return df

test_df = clean(test_df)
test_df = fillna(test_df)
test_df = feature_engineering(test_df)
test_df = feature_encoding(test_df)

In [7]:
#recheck null columns
test_df.columns[test_df.isnull().any()].to_list()

[]

In [8]:
#save the id column and drop it from test_df
test_ids = test_df['Id']
X_test_real = test_df.reindex(columns=X_train.columns, fill_value=0)


In [9]:
#recheck data shape
print(f"Train columns: {X_train.shape[1]}")
print(f"Train columns: {X_test_real.shape[1]}")
print(X_test_real.isnull().sum().sum())

Train columns: 168
Train columns: 168
0


In [10]:
#predict sale price on test set
y_pred = model.predict(X_test_real)
submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": y_pred
})
submission.to_csv("even_better_submission.csv", index = False)
#the "submission.csv" file was from when the hyperparameters weren't tuned

In [ ]:
#submission.csv is the one without hyperparameters tuning, got a RMSLE of 0.14043 on Kaggle
#better_submission got the hyperparameters tuned got a RMSLE of 0.13273 on Kaggle, which is nice
#even better_submission is the one after I removed the outlier, but it only got 0.13503 somehow, shame

In [12]:
import joblib

# Save model
joblib.dump(model, 'best_houseprice_predictor2.joblib')
print("Model saved!")

Model saved!
